In [ ]:
import ee
import geemap
from datetime import datetime
ee.Authenticate()
ee.Initialize(project='satelliteanalysis')

In [ ]:
Map=geemap.Map()

In [ ]:
# Approximate coordinates near Al Zubarah Beach, Khor Fakkan
khor_fakkan = ee.Geometry.Point(56.36, 25.34)
Map.centerObject(khor_fakkan, 11)

In [ ]:
# Use 2-week window around May 19, 2025
start = '2025-05-14'
end = '2025-05-26'

s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(khor_fakkan) \
    .filterDate(start, end) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VV')


    # Get list of acquisition dates
dates = s1.aggregate_array('system:time_start').getInfo()

# Convert and print dates
print("Sentinel-1 SAR Pass Dates over Khor Fakkan:")
for timestamp in dates:
    date = datetime.utcfromtimestamp(timestamp / 1000).strftime('%Y-%m-%d')
    print(f"→ {date}")


Sentinel-1 SAR Pass Dates over Khor Fakkan:
→ 2025-05-16
→ 2025-05-23


In [ ]:
# Median to reduce noise
image = s1.median().clip(khor_fakkan.buffer(20000))

# Oil appears as dark pixels (low backscatter)
oil_mask = image.lt(-22)  # Adjust threshold as needed



In [ ]:
# VV band visualization
vv_vis = {'min': -25, 'max': 0, 'palette': ['black', 'white']}
Map.add_basemap('SATELLITE')
Map.addLayer(image, vv_vis, 'Sentinel-1 VV')
Map.addLayer(oil_mask.updateMask(oil_mask), {'palette': 'cyan'}, 'Potential Oil Spill')
Map.addLayer(khor_fakkan, {}, 'Khor Fakkan Location')
Map


Map(center=[25.34, 56.36000000000001], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=…